# Ullada — Catastro INSPIRE Data Exploration

**Source:** Barcelona bulk GML (Catastro code 08900 = INE 08019)

> In the app, use OVC web services for single queries (WCF services PDF). Bulk file is for exploration only.

## 1. Download & unzip

Run once. Files land in `/tmp/BU_08900/`.

In [6]:
import urllib.request, zipfile, pathlib, ssl, certifi

URL = 'https://www.catastro.hacienda.gob.es/INSPIRE/Buildings/08/08900-BARCELONA/A.ES.SDGC.BU.08900.zip'
OUT_DIR = pathlib.Path('/tmp/BU_08900')
ZIP_PATH = pathlib.Path('/tmp/BU_08900.zip')

ctx = ssl.create_default_context(cafile=certifi.where())  # verified TLS; certifi includes the FNMT root Catastro uses

if not ZIP_PATH.exists():
    print('Downloading (~47 MB)...')
    opener = urllib.request.build_opener(urllib.request.HTTPSHandler(context=ctx))
    with opener.open(URL) as r, open(ZIP_PATH, 'wb') as f:
        f.write(r.read())
    print('Done.')
else:
    print('ZIP already exists, skipping.')

OUT_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(OUT_DIR)
print('Extracted:', [p.name for p in OUT_DIR.iterdir()])

ZIP already exists, skipping.
Extracted: ['A.ES.SDGC.BU.08900.building.gml', 'A.ES.SDGC.BU.08900.building.gfs', 'A.ES.SDGC.BU.MD.08900.xml', 'A.ES.SDGC.BU.08900.buildingpart.gfs', 'A.ES.SDGC.BU.08900.buildingpart.gml', 'A.ES.SDGC.BU.08900.otherconstruction.gml']


## 2. Load building.gml

69,911 buildings in Barcelona city.

In [7]:
import geopandas as gpd

GML = '/tmp/BU_08900/A.ES.SDGC.BU.08900.building.gml'
buildings = gpd.read_file(GML, engine='pyogrio')
print(f'Rows: {len(buildings):,}  CRS: {buildings.crs}')
buildings.head(3)

AttributeError: module 'geopandas' has no attribute 'read_file'

## 3. Fields that actually exist

27 columns — key ones for Ullada:

| Field | Content |
|---|---|
| `reference` | 14-char cadastral ref — use for web service calls |
| `currentUse` | `1_residential`, `4_3_publicServices`, `3_industrial` … |
| `beginning` | Year built (ISO date, year-only resolution) |
| `value` | Gross floor area m² (officialAreaReference=grossFloorArea) |
| `numberOfBuildingUnits` | All units (flats + commercial + storage) |
| `numberOfDwellings` | Residential dwellings only |
| `conditionOfConstruction` | functional / ruin / underConstruction / projected |
| `informationSystem` | URL to Catastro viewer |
| `documentLink` | Direct URL to facade photo (JPEG) |
| `geometry` | Footprint polygon EPSG:25831 |

**`numberOfFloorsAboveGround` is NULL here** — use buildingpart.gml for floor counts.

In [ ]:
for col in buildings.columns:
    sample = buildings[col].dropna().iloc[0] if buildings[col].notna().any() else '—'
    print(f'{col:<45}  {str(sample)[:80]}')

## 4. Known address — Sagrada Família

In [ ]:
from shapely.geometry import Point

# Sagrada Família in EPSG:25831 (from WGS84 41.403629, 2.174355; Catastro address C/ Marina 253)
sagrada = Point(430987, 4583894)
buildings['dist'] = buildings.geometry.distance(sagrada)
row = buildings.nsmallest(1, 'dist').iloc[0]

print(f"Cadastral ref:    {row['reference']}")
print(f"Use:              {row['currentUse']}")
print(f"Year built:       {row['beginning'][:4]}")
print(f"Gross floor area: {row['value']} m²")
print(f"Building units:   {row['numberOfBuildingUnits']}")
print(f"Catastro URL:     {row['informationSystem']}")
print(f"Facade photo:     {row['documentLink']}")

## 5. Floor counts — buildingpart.gml

319,559 parts. `numberOfFloorsAboveGround` lives here.
Link via: `part.localId.split('_part')[0] == building.localId`

In [ ]:
parts = gpd.read_file('/tmp/BU_08900/A.ES.SDGC.BU.08900.buildingpart.gml', engine='pyogrio')
print(f'Parts rows: {len(parts):,}')
print(parts[['localId','numberOfFloorsAboveGround','numberOfFloorsBelowGround']].head(5).to_string())

## 6. Single-building lookup via web service (use this in the app)

```
GET https://ovc.catastro.meh.es/ovcservweb/OVCSWLocalizacionRC/OVCCoordenadas.asmx/Consulta_CPMRC
    ?Provincia=BARCELONA&Municipio=BARCELONA&SRS=EPSG:25831&RC=<14-char-ref>
```

In [ ]:
import urllib.request, ssl, xml.dom.minidom, certifi

RC = '1040501DF3814A'  # Sagrada Família (C/ Marina 253)

ctx = ssl.create_default_context(cafile=certifi.where())  # verified TLS; certifi includes the FNMT root Catastro uses

url = ('https://ovc.catastro.meh.es/ovcservweb/OVCSWLocalizacionRC/'
       f'OVCCoordenadas.asmx/Consulta_CPMRC?Provincia=BARCELONA&Municipio=BARCELONA&SRS=EPSG:25831&RC={RC}')
opener = urllib.request.build_opener(urllib.request.HTTPSHandler(context=ctx))
with opener.open(url) as r:
    xml_bytes = r.read()

pretty = xml.dom.minidom.parseString(xml_bytes).toprettyxml(indent='  ')
print('\n'.join(pretty.splitlines()[:60]))